In [2]:
import pandas as pd
import plotly.express as px
from datetime import datetime
import os
import re 
from tqdm import tqdm

In [3]:
pd.set_option("display.max_colwidth", 100)  # after this you can print any column length
pd.set_option("display.max_rows", 7)  # after this you can print any column length
pd.set_option("display.max_columns", None)  # after this you can print any column length

## Load data 

In [4]:
data_dir = 'data'
motions_data_path = os.path.join(data_dir, 'motions_raw.csv')

if not os.path.exists(motions_data_path): 
    raise FileNotFoundError('Local data file not found. Please run the data collection notebook first.')

df = pd.read_csv(motions_data_path, index_col=0)
df

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2
0,Esta casa prohibiría el empotramiento en el periodismo,"Se denomina ""periodistas empotrados"" a aquellos que, en zonas altamente violentas, acompañan con...",CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ECCQ las comunidades indígenas deberían sabotear los proyectos de infraestructura en su territor...,NaN,CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Esta casa lamenta la fuerte personalización de los movimientos sociales y políticos (LePen en Fr...,NaN,CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8648,THS trends of government-imposed regulations pushing full responsibility on producers for waste ...,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8649,"As a feminist, THR the propagation of sexual liberation as a symbol of progress for gender minor...",NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8650,THBT the outcome of a federal election should always reflect the popular vote,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
len(df['Tournament'].unique()), len(df['Motion'].unique())

(748, 8149)

## Data polishing 

In [6]:
df['Motion'] = df['Motion'].astype("string")
df['Motion'] = df['Motion'].str.replace('\r\n', ' ')
df['Motion'] = df["Motion"].str.strip()

### Remove duplicates (motion used in same tournament across different rounds)
in future, better way of handling would be to accumulate/distinguish between these 

In [7]:
# check for duplicates on motion and round 
# motions can be duplicated (if same is used in two different rounds)
duplicates = df[df.duplicated(subset=['Motion', 'Tournament'], keep=False)]

if len(duplicates) > 0:
    raise ValueError(f"Found {len(duplicates)} duplicates in the data. This likely means the same motion is used in two different rounds.")


ValueError: Found 707 duplicates in the data. This likely means the same motion is used in two different rounds.

In [8]:
duplicates[['Motion', 'round_1', 'round_2', 'Tournament']]

,Motion,round_1,round_2,Tournament
35,THW marry for utility rather than love and passion,Gold Final,NaN,DK Wien Mini Invitational 2023
36,THW marry for utility rather than love and passion,Silver Final,NaN,DK Wien Mini Invitational 2023
38,ECA que está no interesse de movimentos liberais de oposição em regimes democrático-autoritários...,Pré-Semifinal,NaN,Campeonato Mundial de Debate em Língua Portuguesa 2021
...,...,...,...,...
8599,"In developing countries, This House Believes that the provision of public utilities (e.g. electr...",Highschool Semifinals,NaN,Asian Online Debating Championship BP 2023
8644,THW allow police commanders to be criminally liable for instances of police brutality committed ...,Open Semifinals,NaN,World Debate Open 2022
8645,THW allow police commanders to be criminally liable for instances of police brutality committed ...,EFL Semifinals,NaN,World Debate Open 2022


In [9]:
df.drop_duplicates(subset=['Motion', 'Tournament'], keep=False, inplace=True) #FIXME: better to aggregate scores
df

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2
0,Esta casa prohibiría el empotramiento en el periodismo,"Se denomina ""periodistas empotrados"" a aquellos que, en zonas altamente violentas, acompañan con...",CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ECCQ las comunidades indígenas deberían sabotear los proyectos de infraestructura en su territor...,NaN,CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Esta casa lamenta la fuerte personalización de los movimientos sociales y políticos (LePen en Fr...,NaN,CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8648,THS trends of government-imposed regulations pushing full responsibility on producers for waste ...,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8649,"As a feminist, THR the propagation of sexual liberation as a symbol of progress for gender minor...",NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8650,THBT the outcome of a federal election should always reflect the popular vote,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Duplicate motions used across different tournaments

In [10]:
recycled = df[df.duplicated(subset=['Motion'], keep=False)]

In [11]:
df[df['Motion'] == recycled.iloc[0]['Motion']]

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2
51,THBT minority artists should create street art and other art in public spaces rather than art in...,NaN,Netaji Subhas Memorial Parliamentary Debate 2023,NSMPD 2023,https://nsmpd23.calicotab.com/nsmpd23,Round 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1000,THBT minority artists should create street art and other art in public spaces rather than art in...,NaN,The Queen Mary IV 2021,QMIV 2021,https://tabtastic.calicotab.com/qmiv2021,Round 1,P,8.0,16.0,40.0,22.0,19.0,15.0,4861.0,14.0,34.0,21.0,7.0,12.0,28.0,4.0,6.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN
3765,THBT minority artists should create street art and other art in public spaces rather than art in...,NaN,The IITB Grand Prix,The IITB Grand Prix,https://iitbgrandprix.calicotab.com/iitb2024,Round 2,P,10.0,16.0,38.0,32.0,7.0,19.0,6093.0,19.0,45.0,9.0,28.0,7.0,11.0,15.0,18.0,46.0,NaN,NaN,NaN,NaN,NaN,NaN


## Basics

### Tournament year

In [12]:
def get_year(tournament: str):
    pattern = re.compile(r'\d{4}')
    match = pattern.search(tournament)

    if tournament == 'eurosdc-25':
        return 2025

    if match:
        return int(match.group(0))
    else:
        return None

df['Year'] = df['Tournament'].apply(get_year)
df.value_counts('Year')

Year
2024.0    1823
2025.0    1683
2023.0    1384
          ... 
2016.0      42
2018.0      26
2014.0      19
Name: count, Length: 11, dtype: int64

### Motion types

In [13]:
full_to_abbr = {
    'BT': "believes",
    'B': "believes",
    "S": "supports",
    "O": "opposes",
    "W": "would",
    "R": "regrets",
    "P": "prefers",

    "PAW": "prefers a world",
    "P a world": "prefers a world",
    #no one uses these abbrs, just made them up for convenience
    "HOP": "hopes",
    "CON": "condemns",
    "WEL": "welcomes",
    "PRE": "predicts",
    "WIL": "will",
    "CEL": "celebrates",
    "REJ": "rejects",
    "MOU": "mourns",
    "LAM": "laments",
    "EMB": "embraces",
    "DEF": "defends",
}

all_types = set(full_to_abbr.values()) # *most types (this is used to fix up actor motions)
all_types_abbr = set(full_to_abbr.keys())

all_types, all_types_abbr


({'believes',
  'celebrates',
  'condemns',
  'defends',
  'embraces',
  'hopes',
  'laments',
  'mourns',
  'opposes',
  'predicts',
  'prefers',
  'prefers a world',
  'regrets',
  'rejects',
  'supports',
  'welcomes',
  'will',
  'would'},
 {'B',
  'BT',
  'CEL',
  'CON',
  'DEF',
  'EMB',
  'HOP',
  'LAM',
  'MOU',
  'O',
  'P',
  'P a world',
  'PAW',
  'PRE',
  'R',
  'REJ',
  'S',
  'W',
  'WEL',
  'WIL'})

In [14]:
def fix_motions(motion: str):
    australs_pattern = re.compile(r'(A|B|C)\W ') #australs bs that adds A, B, C motion types

    if match := re.match(australs_pattern, motion): 
        motion = motion[match.span()[1]:]
    
    # FIXME: deprecated. revisit actor motions parsing at some point
    # e.g. "This House as the environmental movement would support the use of extremist tactics"
    # https://regex101.com/r/jI2tDT/1
    # actor_pattern = re.compile(r"(?:This (h|H)ouse?|TH),?(?P<actor> as( [a-zA-Z']+)+)" + f" ({'|'.join(all_types)})")

    # if match := re.search(actor_pattern, motion):
    #     actor = match.group('actor')
    #     motion = motion.replace(actor, ',' + actor + ',')

    if motion.startswith('That'):
        motion = 'This House believes that' + motion[4:]
    
    return motion

df['Motion'] = df['Motion'].apply(fix_motions)

In [15]:
# https://regex101.com/r/XB1k1w/4
motion_type_pattern = re.compile(r"(?:(T|t)his (H|h)ouse|TH).*?(?P<type>" 
    + "|".join(all_types) + "|"
    + "|".join(t[0].upper() + t[1:] for t in all_types) + "|"
    + "|".join(all_types_abbr)
+ r")\b")

print(motion_type_pattern)

def extract_motion_header(motion: str): 
    match = re.search(motion_type_pattern, motion)
    return match

re.compile('(?:(T|t)his (H|h)ouse|TH).*?(?P<type>condemns|hopes|predicts|laments|prefers|embraces|regrets|opposes|would|will|celebrates|mourns|rejects|defends|believes|prefers a world|supports|welcomes|Condemns|)


In [16]:
def get_motion_type(motion: str): 
    match = extract_motion_header(motion)

    if match: 
        motion_type = match.group('type')

        if motion_type in all_types_abbr:
            return full_to_abbr.get(motion_type, None)
        else: 
            return f'{motion_type.lower()}'
    else:
        return None

df["Motion Type"] = df["Motion"].apply(get_motion_type)

df[df["Motion Type"].isna()] #could not determine motion type / parse motion for these

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type
0,Esta casa prohibiría el empotramiento en el periodismo,"Se denomina ""periodistas empotrados"" a aquellos que, en zonas altamente violentas, acompañan con...",CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.0,None
1,ECCQ las comunidades indígenas deberían sabotear los proyectos de infraestructura en su territor...,NaN,CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.0,None
2,Esta casa lamenta la fuerte personalización de los movimientos sociales y políticos (LePen en Fr...,NaN,CMUDE 2019,cmude2019,https://cmude2019.calicotab.com/cmude2019,Ronda 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8338,This house prefes a world that developed largely through a confrontational model of development ...,"The confrontational model sees development as driven by conflicts, power struggles, and structur...",Manzar,Manzar,https://manzar.calicotab.com/Manzar,Round 2,P,24.0,39.0,85.0,51.0,57.0,41.0,14344.0,34.0,76.0,44.0,33.0,41.0,107.0,38.0,30.0,68.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8531,That we support memory laws,"A: Memory laws are laws governing the interpretation of historical events (e.g., the European Un...",Macquarie | UTS Easters 2023,Easters 2023,https://easters2023.calicotab.com/easters2023,Round 4,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,None
8589,Reducted,"<font color=""#000000"" face=""Arial""><span style=""font-size: 13.3333px;"">Reducted</span></font>",Heritage Parliamentary Debate 2024,HPD2024,https://hpd.calicotab.com/hpd2024,Semifinals,E,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,None


In [17]:
df = df[df["Motion Type"].notna()]
df

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type
14,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports
15,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports
16,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8648,THS trends of government-imposed regulations pushing full responsibility on producers for waste ...,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.0,supports
8649,"As a feminist, THR the propagation of sexual liberation as a symbol of progress for gender minor...",NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.0,regrets
8650,THBT the outcome of a federal election should always reflect the popular vote,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.0,believes


In [18]:
df.reset_index(drop=True, inplace=True)
df

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type
0,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports
1,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports
2,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7756,THS trends of government-imposed regulations pushing full responsibility on producers for waste ...,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 3,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.0,supports
7757,"As a feminist, THR the propagation of sexual liberation as a symbol of progress for gender minor...",NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 2,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.0,regrets
7758,THBT the outcome of a federal election should always reflect the popular vote,NaN,World Debate Open 2022,WDO 2022,https://midp.calicotab.com/wdo2022,Round 1,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022.0,believes


## Motion counts

In [19]:
by_year = df.groupby('Year')
year_counts = by_year['Motion'].count()

In [20]:
px.line(year_counts, markers=True).show()

## Motion types

### All-time motion types

In [21]:
year_counts = df.value_counts('Motion Type')
print(year_counts)

px.bar(
    year_counts, 
    y='count', 
    title='All-time Motion Types',
    labels={'Motion Type': 'Motion Type'},
    height=600,
    width=1000
).show()

Motion Type
believes    2703
would       1944
supports     923
            ... 
laments        1
mourns         1
rejects        1
Name: count, Length: 18, dtype: int64


### This year's motion types

In [22]:
current_year = datetime.now().year

df_this_year = df[df['Year'] == current_year]
df_this_year.tail()

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type
7731,THBT museums should create galleries dedicated exclusively to art created by women,NaN,Birmingham Pro-Am 2025,Bham Pro-am 2025,https://bhamproam.calicotab.com/bhamproam25,Round 4,P,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,believes
7732,"For health care emergencies requiring collective action (e.g. pandemics, pollution-related illne...",The World Health Organization (WHO) is a specialized agency of the United\nNations responsible f...,Birmingham Pro-Am 2025,Bham Pro-am 2025,https://bhamproam.calicotab.com/bhamproam25,Round 5,P,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,would
7733,THR the norm that romantic love is more important than platonic love,NaN,Birmingham Pro-Am 2025,Bham Pro-am 2025,https://bhamproam.calicotab.com/bhamproam25,Bronze Finals,E,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2025.0,regrets
7734,THW prefer to play as a traitor than a faithful,You are a middle class grandma who has been selected to take part in a popular\nreality show cal...,Birmingham Pro-Am 2025,Bham Pro-am 2025,https://bhamproam.calicotab.com/bhamproam25,Silver Finals,E,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,2025.0,would
7735,"THBT the world will become more, rather than less, empathetic in the future",NaN,Birmingham Pro-Am 2025,Bham Pro-am 2025,https://bhamproam.calicotab.com/bhamproam25,Grand Final,E,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2025.0,believes


In [23]:
year_counts = df_this_year.value_counts('Motion Type')

px.bar(
    year_counts, 
    y='count', 
    title=f'Motion Types for {current_year}',
    labels={'Motion Type': 'Motion Type'},
    height=600,
    width=1000
).show()

### Motion types for earliest year

In [24]:
earliest_year = df['Year'].min()
df_earliest_year = df[(df['Year'] == earliest_year)]

In [25]:
year_counts = df_earliest_year.value_counts('Motion Type')

px.bar(
    year_counts, 
    y='count', 
    title=f'Motion Types for {int(earliest_year)}',
    labels={'Motion Type': 'Motion Type'},
    height=600,
    width=1000
).show()

### Over time

In [26]:
types_yearly = (
    df.groupby(['Year', 'Motion Type'])
    .size()
    .reset_index(name='count')
)

fig = px.line(
    types_yearly,
    x='Year',
    y='count',
    color='Motion Type',
    markers=True,
    title='Count of Each Motion Type by Year',
    labels={'count': 'Count', 'Year': 'Year', 'Motion Type': 'Motion Type'},
    height=600,
    width=1000
)

fig.show()

## Vectorization
NB! This step requires access to a Google Cloud project with billing enabled. If desired, you can sign up for a $300 free trial that will be plenty to cover the costs of vectorization.

Alternatively, all vector space information is already included in the exported file.

### Clean up motions not to clutter with motion types

In [27]:
def clean_motion(motion: str):
    match = extract_motion_header(motion)

    if match is None:
        print(f'[WARN] No header found for motion: {motion}')
        return motion

    header_end = match.span()[1]
    chopped = motion[header_end:].lower()

    filler_words = ['that', 'the', 'it', 'is']

    flag = True
    while flag: 
        flag = False

        for w in filler_words: 
            if chopped.startswith(w): 
                chopped = chopped.replace(w, '')
                flag=True
    
    punctuations = ['.', ',' ';', '\r', '\n']
    for p in punctuations: 
        chopped = chopped.replace(p, '')
    return chopped


In [28]:
df['motion_cleaned'] = df['Motion'].map(clean_motion)
df.head()

/var/folders/4t/nd2tlhnx0jlf0zb72mtxv19w0000gn/T/ipykernel_24345/1151224238.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned
0,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,people adopting an indifferent approach to news
1,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,the use of the ludovico treatment for violent prisoners
2,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes,the use of nostalgia* in political campaigning (*nostalgia refers to sentimental feeling of a...
3,This House would continue to keep his identity a secret,"You are a journalist who has recently uncovered secret, indisputable evidence that a long-though...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 4,P,11.0,20.0,42.0,48.0,16.0,14.0,6690.0,27.0,71.0,20.0,23.0,10.0,24.0,0.0,9.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,continue to keep his identity a secret
4,"THW break up banks* considered ""Too Big to Fail"" *Such as but NOT limited to Goldman Sachs, JP ...",NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 5,P,11.0,16.0,40.0,16.0,23.0,17.0,6743.0,19.0,49.0,25.0,24.0,16.0,32.0,16.0,15.0,33.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,"break up banks* considered ""too big to fail"" *such as but not limited to goldman sachs, jp mor..."


In [ ]:
import vertexai
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"]="../credentials.json"
vertexai.init()

In [ ]:
from vertexai.language_models import TextEmbeddingModel
from typing import Optional

def embed(texts, output_dims: Optional[int] = None):
  try: 
    model_id = 'text-embedding-005'
    model = TextEmbeddingModel.from_pretrained(model_id)
    embeddings = model.get_embeddings(texts, output_dimensionality=output_dims)
  except Exception as e:
    print(f"Error embedding text: {e}")
    return None

  return embeddings

In [40]:
# vertexai allows a max of 250 input elements at once
chunk_size = 250

for start in tqdm(range(0, len(df), chunk_size)):
    end = start + chunk_size
    motion_strings = df['motion_cleaned'].values[start:end]

    embeddings = embed(list(motion_strings))
    df.loc[start:end-1, 'Embedding'] = embeddings #NB! pandas uses an inclusive end index (hence the end-1)

#TODO: maybe better to store each value in its own column? or just use a diff file format specifically for floats
from vertexai.language_models import TextEmbedding
df['Embedding'] = df['Embedding'].map(lambda e: e.values if isinstance(e, TextEmbedding) else None) #get only the numbers

df.head()

  0%|          | 0/32 [00:00<?, ?it/s]/var/folders/4t/nd2tlhnx0jlf0zb72mtxv19w0000gn/T/ipykernel_24345/4012748745.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

100%|██████████| 32/32 [02:23<00:00,  4.49s/it]
/var/folders/4t/nd2tlhnx0jlf0zb72mtxv19w0000gn/T/ipykernel_24345/4012748745.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding
0,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,people adopting an indifferent approach to news,"[-0.07384301722049713, -0.00796386506408453, 0.01903834007680416, 0.011261949315667152, 0.028368..."
1,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,the use of the ludovico treatment for violent prisoners,"[-0.01858511008322239, -0.03655075654387474, 0.03427359461784363, 0.004834757652133703, 0.036956..."
2,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes,the use of nostalgia* in political campaigning (*nostalgia refers to sentimental feeling of a...,"[-0.0262506902217865, -0.02885887399315834, 0.01927250251173973, -0.020929662510752678, 0.030339..."
3,This House would continue to keep his identity a secret,"You are a journalist who has recently uncovered secret, indisputable evidence that a long-though...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 4,P,11.0,20.0,42.0,48.0,16.0,14.0,6690.0,27.0,71.0,20.0,23.0,10.0,24.0,0.0,9.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,continue to keep his identity a secret,"[-0.056684479117393494, 0.014354024082422256, 0.013493139296770096, -0.02825758419930935, -0.060..."
4,"THW break up banks* considered ""Too Big to Fail"" *Such as but NOT limited to Goldman Sachs, JP ...",NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 5,P,11.0,16.0,40.0,16.0,23.0,17.0,6743.0,19.0,49.0,25.0,24.0,16.0,32.0,16.0,15.0,33.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,"break up banks* considered ""too big to fail"" *such as but not limited to goldman sachs, jp mor...","[-0.06157379224896431, -0.008737381547689438, 0.0060244896449148655, -0.03530726954340935, 0.025..."


### Checkpoint: Save data locally

In [41]:
os.makedirs(data_dir, exist_ok=True)
df.to_csv(os.path.join(data_dir, 'motions_with_embeddings.csv'), index=False)

In [42]:
import json 

data_dir = 'data'
df = pd.read_csv(os.path.join(data_dir, 'motions_with_embeddings.csv'))
df['Embedding'] = df['Embedding'].apply(lambda x: json.loads(x))

## Semantic analysis

In [43]:
df.head()

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding
0,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,people adopting an indifferent approach to news,"[-0.07384301722049713, -0.00796386506408453, 0.01903834007680416, 0.011261949315667152, 0.028368..."
1,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,the use of the ludovico treatment for violent prisoners,"[-0.01858511008322239, -0.03655075654387474, 0.03427359461784363, 0.004834757652133703, 0.036956..."
2,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes,the use of nostalgia* in political campaigning (*nostalgia refers to sentimental feeling of a...,"[-0.0262506902217865, -0.02885887399315834, 0.01927250251173973, -0.020929662510752678, 0.030339..."
3,This House would continue to keep his identity a secret,"You are a journalist who has recently uncovered secret, indisputable evidence that a long-though...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 4,P,11.0,20.0,42.0,48.0,16.0,14.0,6690.0,27.0,71.0,20.0,23.0,10.0,24.0,0.0,9.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,continue to keep his identity a secret,"[-0.056684479117393494, 0.014354024082422256, 0.013493139296770096, -0.02825758419930935, -0.060..."
4,"THW break up banks* considered ""Too Big to Fail"" *Such as but NOT limited to Goldman Sachs, JP ...",NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 5,P,11.0,16.0,40.0,16.0,23.0,17.0,6743.0,19.0,49.0,25.0,24.0,16.0,32.0,16.0,15.0,33.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,"break up banks* considered ""too big to fail"" *such as but not limited to goldman sachs, jp mor...","[-0.06157379224896431, -0.008737381547689438, 0.0060244896449148655, -0.03530726954340935, 0.025..."


In [44]:
# spread embeddings across multiple columns

df_embeddings = pd.DataFrame(df['Embedding'].to_list())
df_embeddings.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767
0,-0.073843,-0.007964,0.019038,0.011262,0.028369,0.005056,0.017501,0.000514,-0.020941,-0.008539,-0.040220,-0.053195,-0.001798,0.015831,-0.030157,0.064317,0.005764,-0.032942,0.003710,-0.000364,0.037872,-0.050975,-0.055826,-0.059667,-0.043655,-0.023954,-0.063567,-0.001685,-0.105901,0.060551,-0.049405,-0.020177,-0.007847,-0.004557,-0.017306,0.012912,-0.016171,-0.036152,0.054361,-0.007350,-0.027529,0.049087,0.005522,-0.055691,0.006806,0.003706,-0.062975,-0.036369,0.054996,-0.031712,-0.052674,0.031446,-0.042090,0.020762,-0.029278,0.020303,-0.015323,0.010542,-0.007975,-0.025119,-0.035287,-0.023927,0.030468,-0.029533,0.041485,-0.010945,-0.026160,-0.031065,0.020832,-0.064744,0.046524,0.069570,-0.054738,0.027615,0.010066,0.050194,0.043655,0.075452,-0.014525,-0.035515,0.018691,-0.041089,-0.005923,-0.012899,-0.017210,0.000886,-0.031878,-0.037543,-0.018438,-0.018460,-0.002888,0.063580,0.016330,-0.046503,0.053889,-0.008846,0.000008,0.069713,0.052879,-0.003473,-0.012070,-0.016756,-0.055294,-0.065029,-0.027718,0.029053,-0.006515,0.03296

### Standardize features

In [45]:
def check_distr(df: pd.DataFrame):
    mse_mean = ((df.mean() - 0) ** 2).mean()
    mse_std = ((df.std() - 1) ** 2).mean()

    print(f"Mean: {df.values.mean()}, mse: {mse_mean}")
    print(f"St.d.: {df.values.std()}, mse: {mse_std}")

check_distr(df_embeddings)

Mean: -0.0019215823328559345, mse: 0.000496858765732916
St.d.: 0.036033150879183615, mse: 0.9443012376230078


In [46]:
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(df_embeddings)
df_embeddings = pd.DataFrame(scaled)

check_distr(df_embeddings)

Mean: -1.824377312020975e-18, mse: 1.8942892483241267e-33
St.d.: 1.0, mse: 4.153483307307741e-09


### KMeans

In [64]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

silhouettes = []

k_range = (2, min(len(df_embeddings), 300))

def find_best_n_clusters(start: int = k_range[0], end: int = k_range[1]):
    best_n = dict(
        n=0.0,
        score=-1.0
    )

    for n_clusters in range(start, end):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        kmeans.fit(df_embeddings)

        df_kmeans = pd.DataFrame(kmeans.labels_, columns=['centroid'])

        silhouette_avg = float(silhouette_score(df_embeddings, df_kmeans.values.ravel()))
        silhouettes.append(silhouette_avg)

        if silhouette_avg > best_n['score']:
            best_n['score'] = silhouette_avg
            best_n['n'] = n_clusters
    
    return best_n

best_n_clusters = find_best_n_clusters()

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul

/Users/aleks

In [65]:
print(f"Best n clusters = {best_n_clusters['n']} with a silhouette score of {best_n_clusters['score']}")
fig = px.line(x=range(*k_range), y=silhouettes)

avg = sum(silhouettes) / len(silhouettes)
fig.add_hline(y=avg, line_dash='dash')

Best n clusters = 292 with a silhouette score of 0.04080716300307294


NOTE! This generally shows a pretty weak argument for single-label clustering (the silhouette score is as low as 0.03 when 1.00 is the ideal). 

UPDATE: I ran this for more clusters (up to 300) for the first time and we can see a clear trend up, so perhaps I just wasn't clustering enough...

In [50]:
n_clusters = None
max_k = 0

# select the first value above avg
for i,k in enumerate(silhouettes): 
    if i > 50:
        break
    if k > max_k:
        max_k = k
        n_clusters = i + 2 # + 2 because k >= 2

n_clusters

34

In [51]:
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit(df_embeddings)

labels = kmeans.labels_
centroids = kmeans.cluster_centers_

df_kmeans = pd.DataFrame(labels, columns=['centroid'])
df_kmeans.head()

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul



,centroid
0,9
1,0
2,12
3,1
4,18


In [52]:
# import matplotlib.pyplot as plt
# import matplotlib.cm as cm
# import numpy as np
# from sklearn.metrics import silhouette_samples

# sample_silhouette_values = silhouette_samples(df_embeddings, df_kmeans.values.ravel())

# ax1 = plt.subplot()

# y_lower = 10
# for i in range(n_clusters):
#     # Aggregate the silhouette scores for samples belonging to
#     # cluster i, and sort them
#     ith_cluster_silhouette_values = sample_silhouette_values[df_kmeans.values.ravel() == i]

#     ith_cluster_silhouette_values.sort()

#     size_cluster_i = ith_cluster_silhouette_values.shape[0]
#     y_upper = y_lower + size_cluster_i

#     color = cm.nipy_spectral(float(i) / n_clusters)
#     ax1.fill_betweenx(
#         np.arange(y_lower, y_upper),
#         0,
#         ith_cluster_silhouette_values,
#         facecolor=color,
#         edgecolor=color,
#         alpha=0.7,
#     )

#     # Label the silhouette plots with their cluster numbers at the middle
#     ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

#     # Compute the new y_lower for next plot
#     y_lower = y_upper + 10  # 10 for the 0 samples

# ax1.set_title("The silhouette plot for the various clusters.")
# ax1.set_xlabel("The silhouette coefficient values")
# ax1.set_ylabel("Cluster label")

# # The vertical line for average silhouette score of all the values
# silhouette_avg = silhouette_score(df_embeddings, df_kmeans.values.ravel())
# ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

# ax1.set_yticks([])  # Clear the yaxis labels / ticks
# # ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

### PCA

In [53]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(df_embeddings)
df_pca = pd.DataFrame(pca_embeddings, columns=['pca_1', 'pca_2'])

df_pca.head(), pca_embeddings.shape, pca.explained_variance_ratio_

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_pca.py:606: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_pca.py:606: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_pca.py:606: RuntimeWarning:

invalid value encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning:

invalid value encounte

(      pca_1      pca_2
 0  5.529620   0.197906
 1  1.363312  -5.346092
 2  3.072667   5.002999
 3  4.432867 -10.044457
 4 -6.553212  -3.031091,
 (7759, 2),
 array([0.04043253, 0.02668066]))

In [54]:
# centroids through PCA
pca_centroids = pca.transform(centroids)
pca_centroids.shape

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning:

invalid value encountered in matmul



(34, 2)

In [55]:
df_with_emb = pd.concat([df, df_pca, df_kmeans], axis=1)
df_with_emb.head()

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding,pca_1,pca_2,centroid
0,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,people adopting an indifferent approach to news,"[-0.07384301722049713, -0.00796386506408453, 0.01903834007680416, 0.011261949315667152, 0.028368...",5.529620,0.197906,9
1,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,the use of the ludovico treatment for violent prisoners,"[-0.01858511008322239, -0.03655075654387474, 0.03427359461784363, 0.004834757652133703, 0.036956...",1.363312,-5.346092,0
2,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes,the use of nostalgia* in political campaigning (*nostalgia refers to sentimental feeling of a...,"[-0.0262506902217865, -0.02885887399315834, 0.01927250251173973, -0.020929662510752678, 0.030339...",3.072667,5.002999,12
3,This House would continue to keep his identity a secret,"You are a journalist who has recently uncovered secret, indisputable evidence that a long-though...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 4,P,11.0,20.0,42.0,48.0,16.0,14.0,6690.0,27.0,71.0,20.0,23.0,10.0,24.0,0.0,9.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,continue to keep his identity a secret,"[-0.056684479117393494, 0.014354024082422256, 0.013493139296770096, -0.02825758419930935, -0.060...",4.432867,-10.044457,1
4,"THW break up banks* considered ""Too Big to Fail"" *Such as but NOT limited to Goldman Sachs, JP ...",NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 5,P,11.0,16.0,40.0,16.0,23.0,17.0,6743.0,19.0,49.0,25.0,24.0,16.0,32.0,16.0,15.0,33.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,"break up banks* considered ""too big to fail"" *such as but not limited to goldman sachs, jp mor...","[-0.06157379224896431, -0.008737381547689438, 0.0060244896449148655, -0.03530726954340935, 0.025...",-6.553212,-3.031091,18


In [56]:
import plotly.graph_objects as go

fig = go.Figure()

marker = go.scatter.Marker(
    color=df_with_emb['centroid'],
    colorscale='rainbow',
    size=10,
    opacity=0.7
)

centroid_marker = go.scatter.Marker(
    symbol='x',
    size=20,
    opacity=0.5,
    color='gray'
)

fig.add_trace(go.Scatter(x=df_with_emb['pca_1'], y=df_with_emb['pca_2'], mode='markers', hovertext=df_with_emb['Motion'], marker=marker))
# fig.add_trace(go.Scatter(x=pca_centroids[:, 0], y=pca_centroids[:, 1], mode='markers', marker=centroid_marker))

fig.show()

In [57]:
df_with_emb.sort_values('centroid')[['motion_cleaned', 'centroid']]

,motion_cleaned,centroid
2768,the implementation of the plea bargaining system,0
3067,that the gang member is more likely than the computer software executive to be the mayor’s murd...,0
1454,no longer allow police officers to deliberately lie about the existence of evidence as an inter...,0
...,...,...
2811,rise of nihilism in pop culture,33
7110,that we prefer the production of modernised traditional (fairytales) to the production of new o...,33
2354,that contemporary black film makers should actively create films which embrace blaxploitation,33


### t-SNE

In [58]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=3, random_state=42)
X_tsne = tsne.fit_transform(df_embeddings)

df_tsne = pd.DataFrame(X_tsne, columns=['tsne_1', 'tsne_2', 'tsne_3'])
tsne.kl_divergence_

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning:

invalid value encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning:

divide by zero encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning:

overflow encountered in matmul

/Users/alekseiterin/Documents/personal/debates/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning:

invalid value encountered in matmul

/Users/alekseiteri

2.0388970375061035

In [59]:
df_with_emb2 = pd.concat([df_with_emb, df_tsne], axis=1)

In [60]:
px.scatter_3d(df_with_emb2, x='tsne_1', y='tsne_2', z='tsne_3', hover_data='Motion', color='centroid',color_continuous_scale=px.colors.sequential.Jet)

## Auto-label categories with LLMs

In [61]:
from google import genai

from dotenv import load_dotenv
load_dotenv()

client = genai.Client(vertexai=True, project=os.getenv("VERTEXAI_PROJECT_ID"), location="us-central1")

In [62]:
by_category = df_with_emb2.groupby("centroid")

In [63]:
past_themes = [

]

for centroid, group in by_category:
    motions_list = group['Motion'].to_list()
    all_motions = '\n'.join(motions_list)

    user_prompt = f"""
    You are given a list of motions that are related to a single topic or theme. Your task is to label these motions with a single category that best describes them ALL TOGETHER on average, yet is as specific as possible.

    Return your answer as just an appropriate emoji and the category name (1-3 words), no other text.

    For example, 💰 Government Intervention is good.
    Hypothetical Choices or similarly phrased theme names are bad because they are too vague.
    
    Do not repeat past themes, nor use the same emoji: {past_themes}

    Motions:
    {all_motions}
    """
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=user_prompt
    )

    theme = response.text.strip()
    past_themes.append(theme)

    df_with_emb2.loc[df_with_emb2['centroid'] == centroid, 'category'] = theme

    print(centroid, theme)
    print(motions_list[:3])

0 ⚖️ Criminal Justice Reform
['THS the use of the Ludovico Treatment for violent prisoners', 'This House believes that courts should impose adverse cost orders on parties who refuse reasonable settlement offers and subsequently fail to obtain a better outcome at trial', 'Given a chance to catch Robin Hood and with the certainty that no one will find out if you let him escape, TH, as the Sheriff of Nottingham, would incarcerate Robin Hood']
1 🎭 Moral Dilemmas
['This House would continue to keep his identity a secret', 'THW leave Barbieland to find out about the real world', 'THBT the matriarchal indigenous community should take the offer']
2 ⛔ Government Regulations
['This House Opposes the Quebec language laws.', 'THS the Tour of Duty policy', "TH, as Emmanuel Macron's government would shift to Policy A over Policy B."]
3 🗣️ Movement Strategy
['TH, As a marginalized and agressed upon minority community, would actively support vigilantism in their local communities, as opposed to active

In [66]:
category_counts = df_with_emb2.value_counts('category')
px.bar(category_counts)

In [67]:
for category, label in category_counts.items():
    print(category)

🎭 Moral Dilemmas
🗳️ Electoral Systems
🎭 Cultural Values
🌍 Societal Paradigms
🙏 Religious Institutions
🚺 Feminist Strategy
⛔ Government Regulations
🗣️ Movement Strategy
🤝 International Relations
✊ National Identity
⚔️ Military Intervention
🌱 Environmental Activism
🏆 Sports Governance
⚖️ Criminal Justice Reform
📉 Development Economics
🏢 Business Trends
🔨 Corporate Regulation
📱 Social Media Trends
💔 Romantic Relationships
💸 Economic Redistribution
🇨🇳 China's Geopolitics
🎬 Media Representation
🏦 Monetary Policy
🎓 Higher Education
🏫 Education Policy
🧪 Bioethics & Policy
🇪🇺 European Politics
👨‍👩‍👧‍👦 Parental Roles
📰 Media Governance
🏙️ Urban Development
🤖 Artificial Intelligence
🧑‍💼 Labor Rights
🎨 Artistic Expression
👶 Reproductive Rights


In [68]:
most_common_category = category_counts.idxmax()
df_with_emb2[df_with_emb2['category'] == most_common_category][['Motion', 'motion_cleaned']].sample(10)

,Motion,motion_cleaned
3803,TH supports the widespread use of the Pill of Eternal Moonshine,the widespread use of the pill of eternal moonshine
4610,This House (Novak Djokovic) would retire immediately,retire immediately
7413,"This House, as AOC, would continue to distance themselves from the Squad",continue to distance themselves from the squad
...,...,...
2774,TH as the Dalit community opposes the TTD training program,the ttd training program
6606,THBT that X should go to her dream school right away,that x should go to her dream school right away
686,TH as Achilles would not join the war,not join the war


## Difficulty by speaker scores

In [73]:
df_with_emb2['avg_speaks'] = df_with_emb2['speaks'] / df_with_emb2['n'] / (4 * 2)
df_with_emb2['avg_speaks'].describe()

count    2075.000000
mean       75.838468
std         2.411915
            ...     
50%        75.971154
75%        76.770474
max        81.732143
Name: avg_speaks, Length: 8, dtype: float64

### Sidequest: average speaks across all motions by tournament

This is necessary for us to make inferences based on speaker scores, i.e. we need to know if speaks are really absolutely averaged around 75 in tournaments. 

If so, we can say that a motion with a 73 speaks avg. is more difficult. However, if that tournament's whole average is 73, then it's just normal in difficulty.

In [74]:
non_null_speaks = df_with_emb2[df_with_emb2['speaks'] > 0]

In [75]:
avg_speaks_by_tournament = non_null_speaks.groupby('Tournament')['avg_speaks'].mean()
px.histogram(x=avg_speaks_by_tournament)

In [78]:
sorted_avg_speaks_by_tournament = avg_speaks_by_tournament.sort_values(ascending=False)
sorted_avg_speaks_by_tournament

Tournament
HWS Round Robin 2024             80.771825
HWS Round Robin 2025             80.462500
HWS Round Robin 2023             80.412500
                                   ...    
Southampton Open 2022            70.147321
Wolce The Pre-ABP Debate 2020    64.641369
The Women's Debate 2020          35.166667
Name: avg_speaks, Length: 419, dtype: float64

In [79]:
sorted_avg_speaks_by_tournament.index[0]

'HWS Round Robin 2024'

In [80]:
non_null_speaks[non_null_speaks['Tournament'] == sorted_avg_speaks_by_tournament.index[0]]

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding,pca_1,pca_2,centroid,tsne_1,tsne_2,tsne_3,category,avg_speaks
5097,THW break up the largest asset management firms,"BlackRock, Vanguard, and State Street control over $20 trillion worth of\nassets and primarily i...",HWS Round Robin 2024,HWS RR 2024,https://hwsrr2024.calicotab.com/hwsrr2024,Round 6,P,7.0,6.0,8.0,8.0,13.0,7.0,4515.0,11.0,27.0,20.0,11.0,15.0,39.0,18.0,10.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,would,break up the largest asset management firms,"[-0.03843911364674568, -0.0009442180162295699, 0.07152541726827621, -0.02581784501671791, 0.0054...",-6.676449,-2.960916,18,-21.341045,9.959842,-13.889572,🔨 Corporate Regulation,80.625000
5098,THR the glorification of ethno-nationalist figures associated with pro-democracy movements (e.g....,NaN,HWS Round Robin 2024,HWS RR 2024,https://hwsrr2024.calicotab.com/hwsrr2024,Round 7,P,7.0,11.0,25.0,17.0,12.0,12.0,4517.0,13.0,27.0,20.0,14.0,10.0,26.0,2.0,8.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,regrets,the glorification of ethno-nationalist figures associated with pro-democracy movements (eg aung...,"[0.020924964919686317, 0.003186224028468132, -0.005072903819382191, -0.02098626270890236, 0.0360...",0.042885,8.077652,28,-2.840203,-5.614767,33.597038,✊ National Identity,80.660714
5099,THBT the UK should have provided extensive military support to the independent state of Biafra.,"In 1967, the Igbo people declared Biafra to be a state independent from\nNigeria. Nigeria had re...",HWS Round Robin 2024,HWS RR 2024,https://hwsrr2024.calicotab.com/hwsrr2024,Round 1,P,7.0,11.0,21.0,12.0,16.0,17.0,4514.0,9.0,21.0,9.0,12.0,10.0,24.0,11.0,12.0,32.0,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,believes,the uk should have provided extensive military support to the independent state of biafra,"[-0.0033884106669574976, -0.036689311265945435, 0.0027853362262248993, 0.006915738806128502, 0.0...",-6.058627,1.441269,28,-27.261654,-10.716484,16.003820,✊ National Identity,80.607143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5103,"Assuming it is possible to safely and easily move fetuses into safe artificial wombs, THBT womxn...",NaN,HWS Round Robin 2024,HWS RR 2024,https://hwsrr2024.calicotab.com/hwsrr2024,Round 5,P,7.0,9.0,17.0,14.0,15.0,8.0,4542.0,14.0,32.0,23.0,15.0,12.0,32.0,2.0,7.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,believes,womxn should not have the right to an abortion,"[-0.04030105844140053, -0.022214466705918312, 0.06067390739917755, 0.01282528880983591, 0.031878...",3.494797,4.413315,24,14.262232,-0.250685,12.743176,👶 Reproductive Rights,81.107143
5104,THR the predominance of romantic love in popular culture.,NaN,HWS Round Robin 2024,HWS RR 2024,https://hwsrr2024.calicotab.com/hwsrr2024,Round 8,P,7.0,7.0,11.0,12.0,17.0,2.0,4479.0,13.0,31.0,23.0,12.0,15.0,39.0,11.0,7.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,regrets,the predominance of romantic love in popular culture,"[-0.01366769801825285, -0.00017853894678410143, 0.047990910708904266, -0.06902904808521271, 0.01...",12.464793,3.186395,33,22.885828,3.425421,-14.304794,🎬 Media Representation,79.982143
5105,THP a world where post-colonial states adopted and use Interlanguages over the usage of colonial...,"For the purposes of this debate, an Interlanguage is used for communication\nbetween speakers of...",HWS Round Robin 2024,HWS RR 2024,https://hwsrr2024.calicotab.com/hwsrr2024,Round 9,P,7.0,10.0,22.0,20.0,8.0,10.0,4531.0,14.0,34.0,11.0,19.0,8.0,20.0,9.0,10.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,2024.0,prefers a world,where post-colonial states adopted and use interlanguages over the usage of colonial languages ...,"[-0.024106625467538834, -0.0473753735423088

In [81]:
df_with_emb2['tournament_avg_speaks'] = df_with_emb2['Tournament'].map(avg_speaks_by_tournament)
df_with_emb2[['Tournament', 'tournament_avg_speaks']]

,Tournament,tournament_avg_speaks
0,Cambridge IV 2023,76.606818
1,Cambridge IV 2023,76.606818
2,Cambridge IV 2023,76.606818
...,...,...
7756,World Debate Open 2022,NaN
7757,World Debate Open 2022,NaN
7758,World Debate Open 2022,NaN


### Average speaks

In [82]:
df_with_emb2['speaks'].value_counts()

speaks
2407.0     5
6099.0     4
8583.0     4
          ..
8541.0     1
12620.0    1
4972.0     1
Name: count, Length: 1878, dtype: int64

In [86]:
reliable_with_difficulty = df_with_emb2[(df_with_emb2['avg_speaks'] >= 50) & (df_with_emb2['avg_speaks'] <= 90) & (df_with_emb2['tournament_avg_speaks'] >= 50) & (df_with_emb2['tournament_avg_speaks'] <= 90) & (df_with_emb2['n'] >= 8)]

In [87]:
px.histogram(x=reliable_with_difficulty['avg_speaks'], nbins=20)

In [88]:
reliable_with_difficulty.sort_values(by='avg_speaks', ascending=False)[['avg_speaks', 'Motion', 'Tournament']]

,avg_speaks,Motion,Tournament
2116,78.737500,This House Supports the application of the Gender Blindness Simulator to the world.,Pre-WUDC Challenge 2024: HSDA Fundraiser
2317,78.430556,This house would abolish income tax and replace it with a property tax,Sydney Mini 2024
1082,78.387500,"TH, as Vietnam, would significantly prioritize relations with the West over China",ANU Spring 2023
...,...,...,...
435,68.089286,This House believes that we would ban big game hunting.,Wolce The Pre-ABP Debate 2020
437,64.406250,THS a grassroots movement to Boycott China.,Wolce The Pre-ABP Debate 2020
436,61.428571,This House believes that we would ban plain clothes policing,Wolce The Pre-ABP Debate 2020


In [89]:
reliable_with_difficulty.loc[:, 'diff_avg_speaks'] = reliable_with_difficulty['avg_speaks'] - reliable_with_difficulty['tournament_avg_speaks']
reliable_with_difficulty[['Tournament', 'avg_speaks', 'tournament_avg_speaks', 'diff_avg_speaks']]

/var/folders/4t/nd2tlhnx0jlf0zb72mtxv19w0000gn/T/ipykernel_24345/613685175.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Tournament,avg_speaks,tournament_avg_speaks,diff_avg_speaks
0,Cambridge IV 2023,76.329545,76.606818,-0.277273
1,Cambridge IV 2023,77.261364,76.606818,0.654545
2,Cambridge IV 2023,76.795455,76.606818,0.188636
...,...,...,...,...
7747,Erasmus Rotterdam Open 2024,76.062500,76.381250,-0.318750
7748,Erasmus Rotterdam Open 2024,77.687500,76.381250,1.306250
7749,Erasmus Rotterdam Open 2024,75.375000,76.381250,-1.006250


In [90]:
px.histogram(x=reliable_with_difficulty['diff_avg_speaks'])

In [91]:
reliable_with_difficulty.sort_values(by='diff_avg_speaks', ascending=False)[['Motion', 'Tournament', 'round_1', 'avg_speaks', 'tournament_avg_speaks', 'diff_avg_speaks']]

,Motion,Tournament,round_1,avg_speaks,tournament_avg_speaks,diff_avg_speaks
435,This House believes that we would ban big game hunting.,Wolce The Pre-ABP Debate 2020,Round 1,68.089286,64.641369,3.447917
4221,This House Opposes the adoption of a common currency by ECOWAS.,Pretoria Parley Invitational 2021,Round 5,76.000000,73.622321,2.377679
4218,"This House Prefers a world where major environmental resources (i.e. Amazon Forest, Major Rivers...",Pretoria Parley Invitational 2021,Round 2,75.589286,73.622321,1.966964
...,...,...,...,...,...,...
4217,THW require Facebook to pay royalties to news media organizations for displaying their content.,Pretoria Parley Invitational 2021,Round 1,69.901786,73.622321,-3.720536
184,THS female athletes who capitalize on their sexuality to increase the viewership of female sports,Sofia Online: Send Sofia to Euros Edition,Round 1,72.308824,76.170411,-3.861587
6387,THW heavily tax corporations which automate their workforce,Durham IV 2021,Round 5,68.556818,73.690909,-5.134091


In [92]:
import numpy as np

def sigmoid_difficulty(x: float) -> float:
    k = 5
    a = 0
    b = -2 #FIXME: very arbitrary
    return k / (1 + np.exp(a + x * b))

reliable_with_difficulty.loc[:, 'difficulty'] = sigmoid_difficulty(-reliable_with_difficulty['diff_avg_speaks'])

px.histogram(reliable_with_difficulty['difficulty'], nbins=10)

/var/folders/4t/nd2tlhnx0jlf0zb72mtxv19w0000gn/T/ipykernel_24345/1793620806.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [94]:
reliable_with_difficulty[['Motion', 'Tournament', 'tournament_avg_speaks', 'avg_speaks', 'diff_avg_speaks', 'difficulty']].sort_values(by='difficulty', ascending=False)

,Motion,Tournament,tournament_avg_speaks,avg_speaks,diff_avg_speaks,difficulty
6387,THW heavily tax corporations which automate their workforce,Durham IV 2021,73.690909,68.556818,-5.134091,4.999826
184,THS female athletes who capitalize on their sexuality to increase the viewership of female sports,Sofia Online: Send Sofia to Euros Edition,76.170411,72.308824,-3.861587,4.997789
4217,THW require Facebook to pay royalties to news media organizations for displaying their content.,Pretoria Parley Invitational 2021,73.622321,69.901786,-3.720536,4.997068
...,...,...,...,...,...,...
4218,"This House Prefers a world where major environmental resources (i.e. Amazon Forest, Major Rivers...",Pretoria Parley Invitational 2021,73.622321,75.589286,1.966964,0.095956
4221,This House Opposes the adoption of a common currency by ECOWAS.,Pretoria Parley Invitational 2021,73.622321,76.000000,2.377679,0.042660
435,This House believes that we would ban big game hunting.,Wolce The Pre-ABP Debate 2020,64.641369,68.089286,3.447917,0.005055


### Difficulty by theme

In [95]:
pd.set_option('display.max_columns', None)
by_category_with_difficulty = reliable_with_difficulty.groupby('category')

avg_difficulty_by_category = by_category_with_difficulty['difficulty'].median().sort_values(ascending=True)
px.scatter(avg_difficulty_by_category, x='difficulty', labels={'x': 'Median difficulty', 'y': 'Theme'}, width=1000, height=600, title="Difficulty x Theme").show()

### Difficulty x motion type

In [96]:
by_category_with_difficulty = reliable_with_difficulty.groupby('Motion Type')

avg_difficulty_by_category = by_category_with_difficulty['difficulty'].median().sort_values(ascending=True)
px.bar(avg_difficulty_by_category, width=1000, height=600).show()

### Difficulty x Year

In [98]:
by_year_with_difficulty = reliable_with_difficulty.groupby('Year')

avg_difficulty_by_year = by_year_with_difficulty['difficulty'].median().sort_values(ascending=True)
px.scatter(avg_difficulty_by_year, width=1000, height=600).show()

### Bring results in

In [99]:
df_with_emb2.loc[reliable_with_difficulty.index, 'difficulty'] = reliable_with_difficulty['difficulty']

## Motion balance

In [100]:
with_results = df_with_emb2[(df_with_emb2['n'] >= 10) & (df_with_emb2['sum_og'] > 0)] # only in prelim rounds (technically should check if any of sum_side > 0)

In [101]:
motion = with_results.iloc[0]
motion

Motion                                                                   THS people adopting an indifferent approach to news
Info Slide               For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...
Tournament                                                                                                 Cambridge IV 2023
                                                                        ...                                                 
avg_speaks                                                                                                         76.329545
tournament_avg_speaks                                                                                              76.606818
difficulty                                                                                                          3.175948
Name: 0, Length: 43, dtype: object

In [102]:
assert motion[['sum_og', 'sum_oo', 'sum_cg', 'sum_co']].sum() == motion['n'] * 6, "Does not compute."

In [103]:
import numpy as np 
N = motion['n']
mu = np.array([1.5, 1.5, 1.5])

In [104]:
x_bar = motion[['sum_og', 'sum_oo', 'sum_cg']].to_numpy() / N

In [105]:
var_og = motion['sum_og_og'] / N - (motion['sum_og'] / N) ** 2
var_oo = motion['sum_oo_oo'] / N - (motion['sum_oo'] / N) ** 2
var_cg = motion['sum_cg_cg'] / N - (motion['sum_cg'] / N) ** 2
var_co = motion['sum_co_co'] / N - (motion['sum_co'] / N) ** 2

var_og, var_oo, var_cg, var_co

(np.float64(1.3553719008264458),
 np.float64(1.322314049586777),
 np.float64(1.0578512396694213),
 np.float64(0.9752066115702482))

In [106]:
cov_og_oo = motion['sum_og_oo'] / N - (motion['sum_og'] / N) * (motion['sum_oo'] / N)
cov_og_cg = motion['sum_og_cg'] / N - (motion['sum_og'] / N) * (motion['sum_cg'] / N)
cov_og_co = motion['sum_og_co'] / N - (motion['sum_og'] / N) * (motion['sum_co'] / N)

cov_oo_cg = motion['sum_oo_cg'] / N - (motion['sum_oo'] / N) * (motion['sum_cg'] / N)
cov_oo_co = motion['sum_oo_co'] / N - (motion['sum_oo'] / N) * (motion['sum_co'] / N)

cov_cg_co = motion['sum_cg_co'] / N - (motion['sum_cg'] / N) * (motion['sum_co'] / N)

print("OG - OO: ", cov_og_oo)
print("OG | CG: ", cov_og_cg)
print("OG \ CO: ", cov_og_co)

print("OO / CG: ", cov_oo_cg)
print("OO | CO: ", cov_oo_co)

print("CG - CO: ", cov_cg_co)

OG - OO:  -0.8760330578512396
OG | CG:  -0.16528925619834745
OG \ CO:  -0.3140495867768598
OO / CG:  -0.33884297520661155
OO | CO:  -0.10743801652892548
CG - CO:  -0.553719008264463


In [107]:
# og-og, og-oo, og-cg 
# oo-og, oo-oo, oo-cg
# cg-og, cg-oo, cg-cg

V = np.array([
    [var_og, cov_og_oo, cov_og_cg],
    [cov_og_oo, var_oo, cov_oo_cg],
    [cov_og_cg, cov_oo_cg, var_cg],
])
V

array([[ 1.3553719 , -0.87603306, -0.16528926],
       [-0.87603306,  1.32231405, -0.33884298],
       [-0.16528926, -0.33884298,  1.05785124]])

In [108]:
V_inv = np.linalg.inv(V)
V_inv

array([[1.61670107, 1.2373581 , 0.64895081],
       [1.2373581 , 1.77089783, 0.76057792],
       [0.64895081, 0.76057792, 1.29033368]])

In [109]:
xdiff = x_bar - mu
xdiff, xdiff.shape

(array([np.float64(0.40909090909090917), np.float64(-0.13636363636363646),
        np.float64(-0.3181818181818181)], dtype=object),
 (3,))

In [110]:
xdiff.T, xdiff.T.shape

(array([np.float64(0.40909090909090917), np.float64(-0.13636363636363646),
        np.float64(-0.3181818181818181)], dtype=object),
 (3,))

In [111]:
z = N * xdiff.T @ V_inv @ xdiff
z

np.float64(2.1244625042999656)

In [112]:
# sanity check
motion[['sum_og', 'sum_oo', 'sum_cg', 'sum_co']] / int(motion['n']), motion['n']

(sum_og    1.909091
 sum_oo    1.363636
 sum_cg    1.181818
 sum_co    1.545455
 Name: 0, dtype: object,
 np.float64(11.0))

In [113]:
data = [
    [motion['sum_og'] / motion['n'], motion['sum_oo'] / motion['n']],
    [motion['sum_cg'] / motion['n'], motion['sum_co'] / motion['n']]
]

px.imshow(data, text_auto=True) 

### Now put it into a function

In [114]:
def compute_z(motion): 
    import numpy as np 
    N = motion['n']
    mu = np.array([1.5, 1.5, 1.5])

    x_bar = motion[['sum_og', 'sum_oo', 'sum_cg']].to_numpy() / N

    var_og = motion['sum_og_og'] / N - (motion['sum_og'] / N) ** 2
    var_oo = motion['sum_oo_oo'] / N - (motion['sum_oo'] / N) ** 2
    var_cg = motion['sum_cg_cg'] / N - (motion['sum_cg'] / N) ** 2

    cov_og_oo = motion['sum_og_oo'] / N - (motion['sum_og'] / N) * (motion['sum_oo'] / N)
    cov_og_cg = motion['sum_og_cg'] / N - (motion['sum_og'] / N) * (motion['sum_cg'] / N)

    cov_oo_cg = motion['sum_oo_cg'] / N - (motion['sum_oo'] / N) * (motion['sum_cg'] / N)

    V = np.array([
        [var_og, cov_og_oo, cov_og_cg],
        [cov_og_oo, var_oo, cov_oo_cg],
        [cov_og_cg, cov_oo_cg, var_cg],
    ])

    V_inv = np.linalg.inv(V)

    z = N * (x_bar - mu).T @ V_inv @ (x_bar - mu) # need to brush up on this

    return z


In [115]:
z_numpy = compute_z(motion)
z_numpy

np.float64(2.1244625042999656)

### No numpy / raw python 

In [116]:
# def compute_z_no_numpy(motion):
N = motion['n']
mu = [1.5, 1.5, 1.5]

x_bar = motion[['sum_og', 'sum_oo', 'sum_cg']] / N
x_bar = x_bar.values

var_og = motion['sum_og_og'] / N - (motion['sum_og'] / N) ** 2
var_oo = motion['sum_oo_oo'] / N - (motion['sum_oo'] / N) ** 2
var_cg = motion['sum_cg_cg'] / N - (motion['sum_cg'] / N) ** 2

cov_og_oo = motion['sum_og_oo'] / N - (motion['sum_og'] / N) * (motion['sum_oo'] / N)
cov_og_cg = motion['sum_og_cg'] / N - (motion['sum_og'] / N) * (motion['sum_cg'] / N)

cov_oo_cg = motion['sum_oo_cg'] / N - (motion['sum_oo'] / N) * (motion['sum_cg'] / N)

V = [
    [var_og, cov_og_oo, cov_og_cg],
    [cov_og_oo, var_oo, cov_oo_cg],
    [cov_og_cg, cov_oo_cg, var_cg],
]

In [117]:
def determinant_3x3(matrix):
    """Calculate determinant of 3x3 matrix"""
    a, b, c = matrix[0]
    d, e, f = matrix[1]
    g, h, i = matrix[2]
    
    return a*(e*i - f*h) - b*(d*i - f*g) + c*(d*h - e*g)

def invert_matrix_manual(matrix):
    """Manually invert a 3x3 matrix using adjugate method"""
    # Calculate determinant
    det = determinant_3x3(matrix)
    
    if abs(det) < 1e-10:
        raise ValueError("Matrix is singular and cannot be inverted")
    
    # Extract elements
    a, b, c = matrix[0]
    d, e, f = matrix[1]
    g, h, i = matrix[2]
    
    # Calculate cofactor matrix
    cofactor = [
        [+(e*i - f*h), -(d*i - f*g), +(d*h - e*g)],
        [-(b*i - c*h), +(a*i - c*g), -(a*h - b*g)],
        [+(b*f - c*e), -(a*f - c*d), +(a*e - b*d)]
    ]
    
    # Transpose cofactor matrix to get adjugate
    adjugate = [[cofactor[j][i] for j in range(3)] for i in range(3)]
    
    # Multiply by 1/det
    inverse = [[adjugate[i][j] / det for j in range(3)] for i in range(3)]
    
    return inverse



def dot_product(A, B):
    result = 0
    for i in range(len(A)):
        result += A[i] * B[i]
    return result

def matvecmul(A, B):
    result = [0 for _ in range(len(B))]

    for i in range(len(B)):
        result[i] = dot_product(A[i], B)

    return result

In [118]:
V_inv = invert_matrix_manual(V)
V_inv

[[np.float64(1.6167010663914705),
  np.float64(1.2373581011351922),
  np.float64(0.6489508083935341)],
 [np.float64(1.2373581011351922),
  np.float64(1.7708978328173384),
  np.float64(0.760577915376678)],
 [np.float64(0.6489508083935341),
  np.float64(0.760577915376678),
  np.float64(1.2903336773305818)]]

In [119]:
xdiff = x_bar - mu
x_bar, xdiff

(array([np.float64(1.9090909090909092), np.float64(1.3636363636363635),
        np.float64(1.1818181818181819)], dtype=object),
 array([np.float64(0.40909090909090917), np.float64(-0.13636363636363646),
        np.float64(-0.3181818181818181)], dtype=object))

In [120]:
z_temp = matvecmul(V_inv, xdiff)
z_temp

[np.float64(0.2861627106983145),
 np.float64(0.02270381836945312),
 np.float64(-0.24879600963192267)]

In [121]:
z_manual = dot_product(xdiff, z_temp) * N 
z_manual

np.float64(2.124462504299965)

In [122]:
assert (z_numpy - z_manual) < 1e-10, "The two methods of computing z give different results"

### Apply calculation

In [123]:
def try_compute_z(motion): 
    try: 
        return compute_z(motion)
    except Exception as e: 
        print(e)
        print(motion[['sum_og', 'sum_oo', 'sum_cg', 'sum_co', 'n']])
        return None

In [124]:
df_with_emb2.loc[with_results.index, 'z'] = with_results.apply(try_compute_z, axis=1)


In [125]:
df_with_results = df_with_emb2.loc[with_results.index]
fig = px.histogram(df_with_results[df_with_results['z'] < 100], x='z', nbins=100)
fig.show()

In [126]:
# sourced from: https://www.danielsoper.com/statcalc/calculator.aspx?id=12
CRITICAL_VALUES = [
    # (maximum value, level of significance as percentage string, evidence strength)
    (16.26623620, '0.1%', "extremely strong evidence"),
    (11.34486673,    '1%', "strong evidence"),
    (9.83740931,    '2%', "moderate evidence"),
    (7.81472790,    '5%', "weak evidence"),
    (6.25138863,   '10%', "very weak evidence"),
    (2.36597388,   '50%', "extremely weak evidence"),
    (0, '100%', "no evidence")
]

In [127]:
for x_val, significance, evidence_strenth in CRITICAL_VALUES:
    fig.add_vline(
        x=x_val, 
        line_dash="dot", 
        line_color="gray",
        line_width=1,
        annotation_text=evidence_strenth,
        annotation_position="bottom",
        annotation_font_size=12
    )

fig.show()

In [128]:
def get_evidence(motion):
    for critical, level_str, evidence_strength in CRITICAL_VALUES:
        if motion['z'] > critical:
            return evidence_strength
    else: 
        return "no evidence"

In [129]:
df_with_emb2.loc[df_with_results.index, 'imbalance'] = df_with_results.apply(get_evidence, axis=1)
df_with_emb2.head()

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding,pca_1,pca_2,centroid,tsne_1,tsne_2,tsne_3,category,avg_speaks,tournament_avg_speaks,difficulty,z,imbalance
0,THS people adopting an indifferent approach to news,"For the purpose of this debate, the “indifferent approach” means that people can choose to stop ...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 1,P,11.0,21.0,55.0,19.0,23.0,29.0,6717.0,15.0,35.0,14.0,22.0,13.0,27.0,14.0,17.0,37.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,people adopting an indifferent approach to news,"[-0.07384301722049713, -0.00796386506408453, 0.01903834007680416, 0.011261949315667152, 0.028368...",5.529620,0.197906,9,-2.504657,12.766487,-21.688606,📱 Social Media Trends,76.329545,76.606818,3.175948,2.124463,no evidence
1,THS the use of the Ludovico Treatment for violent prisoners,"For the purpose of this debate, the Ludovico Treatment forces people to develop a natural aversi...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 2,P,11.0,24.0,64.0,25.0,27.0,28.0,6799.0,12.0,22.0,16.0,9.0,17.0,43.0,16.0,13.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,the use of the ludovico treatment for violent prisoners,"[-0.01858511008322239, -0.03655075654387474, 0.03427359461784363, 0.004834757652133703, 0.036956...",1.363312,-5.346092,0,12.198110,5.683858,26.094578,⚖️ Criminal Justice Reform,77.261364,76.606818,1.063195,11.034459,moderate evidence
2,This House opposes the use of nostalgia* in political campaigning (*nostalgia refers to sentim...,NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 3,P,11.0,21.0,51.0,32.0,28.0,15.0,6758.0,18.0,36.0,31.0,9.0,19.0,49.0,6.0,8.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,opposes,the use of nostalgia* in political campaigning (*nostalgia refers to sentimental feeling of a...,"[-0.0262506902217865, -0.02885887399315834, 0.01927250251173973, -0.020929662510752678, 0.030339...",3.072667,5.002999,12,20.927673,-3.510602,-14.438703,🗳️ Electoral Systems,76.795455,76.606818,2.033924,7.666420,very weak evidence
3,This House would continue to keep his identity a secret,"You are a journalist who has recently uncovered secret, indisputable evidence that a long-though...",Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 4,P,11.0,20.0,42.0,48.0,16.0,14.0,6690.0,27.0,71.0,20.0,23.0,10.0,24.0,0.0,9.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,continue to keep his identity a secret,"[-0.056684479117393494, 0.014354024082422256, 0.013493139296770096, -0.02825758419930935, -0.060...",4.432867,-10.044457,1,14.225636,23.514471,3.701983,🎭 Moral Dilemmas,76.022727,76.606818,3.814081,35.781977,extremely strong evidence
4,"THW break up banks* considered ""Too Big to Fail"" *Such as but NOT limited to Goldman Sachs, JP ...",NaN,Cambridge IV 2023,Cambridge IV 2023,https://cambridgeiv2023.calicotab.com/_,Round 5,P,11.0,16.0,40.0,16.0,23.0,17.0,6743.0,19.0,49.0,25.0,24.0,16.0,32.0,16.0,15.0,33.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,would,"break up banks* considered ""too big to fail"" *such as but not limited to goldman sachs, jp mor...","[-0.06157379224896431, -0.008737381547689438, 0.0060244896449148655, -0.03530726954340935, 0.025...",-6.553212,-3.031091,18,-21.051291,11.013060,-12.418249,🔨 Corporate Regulation,76.625000,76.606818,2.454550,0.595395,no evidence


In [130]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
df_with_results.sort_values(by='z', ascending=True)

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding,pca_1,pca_2,centroid,tsne_1,tsne_2,tsne_3,category,avg_speaks,tournament_avg_speaks,difficulty,z
95,This House believes that the United States should legalize cocaine,NaN,McGill IV 2025,mcgilliv25,https://mdu.calicotab.com/mcgilliv25,Round 5,P,17.0,26.0,56.0,33.0,35.0,32.0,10250.0,25.0,57.0,28.0,32.0,26.0,66.0,27.0,25.0,59.0,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,believes,that the united states should legalize cocaine,"[-0.047857318073511124, -0.0075638554990291595, 0.06523004919290543, 0.030120601877570152, 0.04004112258553505, 0.04386397451162338, 0.021709224209189415, 0.055467639118433, 0.024144018068909645, -0.04468095675110817, 0.020601466298103333, -0.003437852021306753, -0.013248234987258911, 0.013511142693459988, -0.0036109129432588816, 0.01964840292930603, 0.01949922926723957, 0.005346681457012892, 0.015240279957652092, -0.02087309956550598, 0.05399949848651886, -0.07710430771112442, 0.07245242595672607, 0.013494816608726978, 0.023621374741196632, 0.003437250619754195, -0.030416665598750114, -0.04068945720791817, -0.05565851926803589, 0.06949593126773834, 0.038561996072530746, 0.04015917703509331, 0.030165206640958786, -0.07155495882034302, 0.051133669912815094, 0.05122829228639603, 0.029292531311511993, 0.061640769243240356, -0.014435622841119766, -0.04532907158136368, -0.004596023354679346, -0.008511574007570744, -0.08195208013057709, -0.03531602770090103, 0.00889185257256031, 0.06370468437671661, -0.0053398096933960915, 0.006711494643241167, 0.02423647791147232, -0.05312800034880638, -0.016900572925806046, 0.02809666283428669, -0.05740511417388916, 0.05697642266750336, -0.001878112554550171, -0.012814333662390709, 0.006637547630816698, 0.07462487369775772, -0.007465355563908815, 0.05236115679144859, -0.0024040674325078726, -0.0429806150496006, -0.05916411057114601, 0.03424639254808426, -0.03058847039937973, 0.04157302528619766, 0.018070461228489876, 0.013216594234108925, -0.02950938232243061, -0.056129392236471176, -0.0010674778604879975, -0.011695543304085732, 0.02273252233862877, 0.04964624345302582, -0.006036628969013691, 0.0034572742879390717, 0.010953512042760849, 0.11735697090625763, 0.04434259235858917, 0.03727726638317108, -0.03342136740684509, -0.058221444487571716, 0.01791542023420334, 0.03741960972547531, -0.0377245731651783, 0.001247259322553873, -0.06756783276796341, -0.03514711186289787, 0.012059403583407402, -0.0033396240323781967, -0.0010637086816132069, 0.007223845459520817, 0.012175653129816055, -0.07056514918804169, 0.0350763164460659, 0.01956147700548172, 0.013932798057794571, 0.04483533650636673, 0.01537453942000866, -0.027101323008537292, ...]",-1.694247,-2.828446,30,-11.329549,12.364662,3.113539,🧪 Bioethics & Policy,75.367647,75.310294,2.356775,0.032319
918,THS open borders.,NaN,UCLA Intervarsity Debating Champsionships 2023,UCLA IV 2023,https://ucladebateunion.calicotab.com/uclaiv2023,Round 1,P,15.0,23.0,53.0,25.0,27.0,33.0,9164.0,22.0,54.0,26.0,27.0,22.0,52.0,27.0,23.0,51.0,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,supports,open borders,"[-0.023759551346302032, -0.0630248636007309, -0.014825044199824333, -0.011085213162004948, 0.010093712247908115, 0.015641914680600166, 0.04544640704989433, 0.0027212430723011494, 0.008822476491332054, 0.05818643420934677, 0.01541330385953188, -0.11003290116786957, -0.014108622446656227, -0.02699398808181286, 0.0012676878832280636, 0.03751135244965553, 0.034602176398038864, -0.039271626621484756, 0.0068291109055280685, -0.021160846576094627, 0.04290268197655678, -0.07614955306053162, 0.0024398888926953077, -0.03586803376674652, 0.00696150679141283, -0.04962443187832832, -0.01645326055586338, 0.0074044158682227135, -0.06329207867383957, -0.024177415296435356, -0.01499606017023325, 0.04383324

In [131]:
critical_index = list(map(lambda x: x[2], CRITICAL_VALUES))
critical_index

['extremely strong evidence',
 'strong evidence',
 'moderate evidence',
 'weak evidence',
 'very weak evidence',
 'extremely weak evidence',
 'no evidence']

In [132]:
by_imbalance = df_with_emb2['imbalance'].value_counts().reindex(reversed(critical_index))
by_imbalance

imbalance
no evidence                  420
extremely weak evidence      494
very weak evidence           135
weak evidence                108
moderate evidence             66
strong evidence              143
extremely strong evidence    180
Name: count, dtype: int64

In [134]:
px.bar(by_imbalance, width=1000, height=600).show()

The graphs below are not well-studied, not well-justified and are mostly experimental. Read at your own risk.

### Imbalance x Difficulty

In [140]:
px.scatter(df_with_results, x='z', y='difficulty')

### Imbalance x Theme

In [141]:
px.scatter(df_with_results, x='z', y='category')

In [142]:
imbalance_by_theme = df_with_results.groupby('category')['z'].median()
imbalance_by_theme = imbalance_by_theme.sort_values(ascending=True)

# Define custom y labels at certain y= horizontal locations
import plotly.graph_objects as go

fig = px.bar(
    imbalance_by_theme, 
    x=imbalance_by_theme.index, 
    y=imbalance_by_theme.values, 
    width=1000, 
    height=600,
    title="Median Imbalance by Theme",
    labels={'x': 'Theme', 'y': 'Median Imbalance'},
    # y_label="Median Imbalance"
)

# Add custom y labels at specific y values

for y_val, significance, evidence_strenth in CRITICAL_VALUES:
    fig.add_hline(
        y=y_val, 
        line_dash="dot", 
        line_color="gray",
        line_width=1,
        annotation_text=evidence_strenth,
        annotation_position="right",
        annotation_font_size=12
    )

fig.show()
# px.bar(imbalance_by_theme, x=imbalance_by_theme.index, y=imbalance_by_theme.values, width=1000, height=600).show()

### Imbalance x Type

In [143]:
imbalance_by_theme = df_with_results.groupby('Motion Type')['z'].median()
imbalance_by_theme = imbalance_by_theme.sort_values(ascending=True)

# Define custom y labels at certain y= horizontal locations
import plotly.graph_objects as go

fig = px.bar(
    imbalance_by_theme, 
    x=imbalance_by_theme.index, 
    y=imbalance_by_theme.values, 
    width=1000, 
    height=600,
    title="Median Imbalance by Type",
    labels={'x': 'Type', 'y': 'Median Imbalance'},
    # y_label="Median Imbalance"
)

# Add custom y labels at specific y values

for y_val, significance, evidence_strenth in CRITICAL_VALUES:
    fig.add_hline(
        y=y_val, 
        line_dash="dot", 
        line_color="gray",
        line_width=1,
        annotation_text=evidence_strenth,
        annotation_position="right",
        annotation_font_size=12
    )

fig.show()
# px.bar(imbalance_by_theme, x=imbalance_by_theme.index, y=imbalance_by_theme.values, width=1000, height=600).show()

### Imbalance x Year

In [144]:
imbalance_by_theme = df_with_results.groupby('Year')['z'].median()
imbalance_by_theme = imbalance_by_theme.sort_values(ascending=True)

# Define custom y labels at certain y= horizontal locations
import plotly.graph_objects as go

fig = px.scatter(
    imbalance_by_theme, 
    x=imbalance_by_theme.index, 
    y=imbalance_by_theme.values, 
    width=1000, 
    height=600,
    title="Median Imbalance by Year",
    labels={'x': 'Year', 'y': 'Median Imbalance'},
    # y_label="Median Imbalance"
)

# Add custom y labels at specific y values

for y_val, significance, evidence_strenth in CRITICAL_VALUES:
    fig.add_hline(
        y=y_val, 
        line_dash="dot", 
        line_color="gray",
        line_width=1,
        annotation_text=evidence_strenth,
        annotation_position="right",
        annotation_font_size=12
    )

fig.show()
# px.bar(imbalance_by_theme, x=imbalance_by_theme.index, y=imbalance_by_theme.values, width=1000, height=600).show()

## Export for website

In [145]:
df_with_emb2.to_json('data/motions.json', orient='records') #FIXME: could be a nicer export (remove redudancy, group things together)

## Side balance

probably deserves a separate notebook

In [146]:
df_with_emb2.sample(1)

,Motion,Info Slide,Tournament,short_name,tab_url,round_1,stage_1,n,sum_og,sum_og_og,sum_og_oo,sum_og_cg,sum_og_co,speaks,sum_oo,sum_oo_oo,sum_oo_cg,sum_oo_co,sum_cg,sum_cg_cg,sum_cg_co,sum_co,sum_co_co,oo_wins,cg_wins,co_wins,og_wins,round_2,stage_2,Year,Motion Type,motion_cleaned,Embedding,pca_1,pca_2,centroid,tsne_1,tsne_2,tsne_3,category,avg_speaks,tournament_avg_speaks,difficulty,z,imbalance
844,"THR the concept of ""female empowerment"" in religious texts","Female empowerment in religious texts highlights interpretations that\nemphasize women’s agency, leadership, and equality within spiritual,\nreligious, and communal contexts. It advocates for recognizing women's roles\nas essential to the life depicted in these sacred writings.",48th Mindanao Parliamentary Debate Championship,48th MPDC,https://48thmpdc.calicotab.com/48thmpdc,Round 3,P,13.0,19.0,49.0,24.0,21.0,20.0,7812.0,20.0,42.0,25.0,29.0,18.0,38.0,24.0,21.0,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,regrets,"the concept of ""female empowerment"" in religious texts","[0.011477435007691383, 0.0006584492512047291, 0.030964307487010956, -0.009174715727567673, 0.02234555222094059, 0.0016596298664808273, -0.05918056517839432, 0.03436857461929321, -0.004237263463437557, -0.009582524187862873, -0.06215953081846237, -0.04787730798125267, 0.028096430003643036, -0.041595250368118286, -0.001336197485215962, 0.05929724872112274, 0.05611569434404373, -0.005274635273963213, -0.0048685623332858086, -0.02226913906633854, 0.007841396145522594, -0.06488960981369019, -0.05734361335635185, 0.051496509462594986, 0.03140297532081604, -0.061424870043992996, -0.0910329595208168, -0.033939067274332047, -0.04308052733540535, -0.01620832085609436, 0.003582078730687499, 0.04204433411359787, 0.01015576533973217, 0.04380318149924278, 0.07317159324884415, 0.01635315828025341, 0.06843041628599167, 0.030205372720956802, -0.006150678265839815, 0.012611395679414272, 0.027777304872870445, 0.010917125269770622, -0.026047358289361, -0.012294708751142025, 0.0005567394546233118, 0.05423346906900406, 0.020333673804998398, 0.009045775048434734, -0.024858564138412476, 0.021011587232351303, 0.028607191517949104, -0.02442236803472042, -0.06700775027275085, 0.03719286248087883, 0.027878953143954277, 0.056917380541563034, -0.034154463559389114, -0.01706596650183201, -0.011009369976818562, 0.04969178885221481, -0.02216309681534767, -0.02218257449567318, -0.02346113882958889, -0.042869310826063156, -0.009486804716289043, 0.03069988824427128, 0.025132272392511368, -0.0017856777412816882, -0.04569999873638153, 0.01980200596153736, 0.02889065630733967, -0.05513977259397507, -0.015583527274429798, 0.04092433676123619, -0.015981623902916908, -0.042659346014261246, 0.019692322239279747, 0.07709180563688278, 0.010270428843796253, 0.0241167601197958, -0.00951550155878067, 0.012565416283905506, 0.028410062193870544, -0.019000519067049026, 0.0019491969142109156, -0.01046772487461567, 0.009959287010133266, 0.004154047463089228, 0.008853117935359478, -0.014186427928507328, -0.019079573452472687, -0.01740913838148117, -0.03231640160083771, -0.01631276123225689, 0.052356161177158356, 0.010817823931574821, 0.05178743228316307, 0.06497924774885178, 0.01186365820467472, 0.00027443518047221005, ...]",8.764204,7.231414,13,14.146564,-12.688302,-32.06184,🚺 Feminist Strategy,75.115385,75.283654,2.916747,0.240135,no evidence


In [147]:
og_points = df_with_emb2['sum_og'].sum()
oo_points = df_with_emb2['sum_oo'].sum()
cg_points = df_with_emb2['sum_cg'].sum()
co_points = df_with_emb2['sum_co'].sum()

og_points, oo_points, cg_points, co_points

(np.float64(54773.0),
 np.float64(61785.0),
 np.float64(51519.0),
 np.float64(53269.0))

In [148]:
data = [
    [og_points, oo_points],
    [cg_points, co_points]
]

In [149]:
px.imshow(data, text_auto=True)

In [150]:
og_wins = df_with_emb2['og_wins'].sum()
oo_wins = df_with_emb2['oo_wins'].sum()
cg_wins = df_with_emb2['cg_wins'].sum()
co_wins = df_with_emb2['co_wins'].sum()


all_wins = og_wins + oo_wins + cg_wins + co_wins

og_percent_wins = og_wins / all_wins
oo_percent_wins = oo_wins / all_wins
cg_percent_wins = cg_wins / all_wins
co_percent_wins = co_wins / all_wins

data = [
    [og_percent_wins, oo_percent_wins],
    [cg_percent_wins, co_percent_wins]
]

px.imshow(data, text_auto=True) 